# S7a check — real photos with tape values (TrainingDataPro sample, D-053)

Runs the whole chain (photo → outline → model → cm) on a small public sample of real front/side photos with
hand-taped values. Run the cells **top to bottom**. The photos stay on your Drive: never commit them, never
use them for training (licence CC BY-NC-ND). **Before saving:** Edit → Clear all outputs (POLICY 6).

## 1. Drive and BodyM data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)
if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code

In [ ]:
BRANCH = "s7a-fix-impossible-values"   # set to "main" once PR #13 is merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e "/content/repo[photos]" huggingface_hub
!git -C /content/repo log -1 --format="commit %h  %s"

import sys
sys.path.insert(0, "/content/repo/src")
os.environ["TORCH_HOME"] = f"{DRIVE}/torch_cache"   # keep the tool weights between sessions
RUN = f"{DRIVE}/runs/r4"
TDP = f"{DRIVE}/tdp_sample"

## 3. Is the model itself fine? (one BodyM test silhouette, seconds)
Expect ordinary numbers (e.g. chest 80–120 cm). Impossible numbers here = the model file is the problem.
**Paste back:** the table.

In [ ]:
import glob
f = sorted(glob.glob(f"{DATA_DIR}/testB/mask/*.png"))[0]; s = f.replace("/mask/", "/mask_left/")
!anthro predict-silhouettes --front "{f}" --side "{s}" --height 170 --sex male --ckpt {RUN}/best.pt --data {DATA_DIR}

## 4. Download the sample (once; skipped if already on Drive)

In [ ]:
from huggingface_hub import snapshot_download
if not glob.glob(f"{TDP}/**/*.json", recursive=True):
    snapshot_download("TrainingDataPro/body-measurements-dataset", repo_type="dataset", local_dir=TDP)
print(len(glob.glob(f"{TDP}/**/*.json", recursive=True)), "people")

## 5. Build the pairs files (adults only)
Tape columns use the app's codes. Leg length, pelvis, front build and neck-to-waist are left out: their
definitions are unclear. **Paste back:** the table and the last line.

In [ ]:
import json, pandas as pd
MAP = {"chest_circumference_cm": "chest_girth", "under_chest_circumference_cm": "underbust_girth",
       "waist_circumference_cm": "waist_girth", "hips_circumference_cm": "hip_girth",
       "neck_circumference_cm": "neck_girth", "shoulder_width_cm": "shoulder_width",
       "thigh_circumference_cm": "thigh_girth", "arm_length_cm": "arm_length"}
BINS = [(18, 24, "18-24"), (25, 34, "25-34"), (35, 44, "35-44"), (45, 54, "45-54"), (55, 64, "55-64")]
def age_range(a):
    if a < 18:
        return None                     # under 18: excluded below (adults only)
    return next((lab for lo, hi, lab in BINS if lo <= a <= hi), "65+")

rows = []
for n, j in enumerate(sorted(glob.glob(f"{TDP}/**/*.json", recursive=True)), 1):
    d, folder = json.load(open(j)), os.path.dirname(j)
    imgs = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    fr = [f for f in imgs if "front" in f.lower()]; sd = [f for f in imgs if "side" in f.lower()]
    if len(fr) != 1 or len(sd) != 1:
        print("check folder", n, "files:", imgs); continue
    sex = str(d["gender"]).lower()
    r = {"person": f"T{n:02d}", "front": f"{folder}/{fr[0]}", "side": f"{folder}/{sd[0]}",
         "height_cm": float(d["height"]), "sex": sex, "weight_kg": float(d["weight"]),
         "age_range": age_range(int(d["age"])), "race": d["race"]}
    for k, code in MAP.items():
        r["bust_girth" if (code == "chest_girth" and sex == "female") else code] = d.get(k)
    rows.append(r)
df = pd.DataFrame(rows)
df = df[df.age_range.notna()]
print(df[["person", "height_cm", "weight_kg", "sex", "age_range", "race"]].to_string(index=False))
df.drop(columns="race").to_csv(f"{TDP}/pairs_all.csv", index=False)
df[df.race == "black"].drop(columns="race").to_csv(f"{TDP}/pairs_black.csv", index=False)
print(len(df), "adults;", (df.race == "black").sum(), "Black")

## 5b. Your pose groups
Fill in after looking at the photos (cell 6): **good** = arms away from the body and fitted clothing; **poor** = arms touching the body or very loose clothing. Leave anyone unclear out.

In [ ]:
GOOD = ["T05", "T21"]          # edit
POOR = ["T07", "T12"]          # edit
for name, who in (("good", GOOD), ("poor", POOR)):
    df[df.person.isin(who)].drop(columns="race").to_csv(f"{TDP}/pairs_{name}.csv", index=False)
    print(name, len(who), "people")

## 6. Look at one person's photos (pose check; shown here only)
Front: arms about 45° away? Side: arms down? Whole body in frame? Tight or loose clothing?
Change the code to look at others. **Tell me** what you see; do not paste the photos.

In [ ]:
from PIL import Image, ImageOps
from IPython.display import display
r = df[df.person == "T21"].iloc[0]
for v in ("front", "side"):
    im = ImageOps.exif_transpose(Image.open(r[v])); im.thumbnail((400, 400))
    print(r.person, v); display(im)

## 7. Both tools on everyone, then on the Black participants (a few minutes)
**Paste back:** everything printed (numbers only).

In [ ]:
!anthro compare-segmenters --pairs {TDP}/pairs_all.csv --ckpt {RUN}/best.pt --data {DATA_DIR} --out {TDP}/compare_all.json
!anthro compare-segmenters --pairs {TDP}/pairs_black.csv --ckpt {RUN}/best.pt --data {DATA_DIR} --out {TDP}/compare_black.json
!anthro compare-segmenters --pairs {TDP}/pairs_good.csv --ckpt {RUN}/best.pt --data {DATA_DIR} --out {TDP}/compare_good.json
!anthro compare-segmenters --pairs {TDP}/pairs_poor.csv --ckpt {RUN}/best.pt --data {DATA_DIR} --out {TDP}/compare_poor.json

## 8. One person's full result
Change `WHO` and `TOOL` (the winner above). **Paste back:** the result and the tape row printed under it.

In [ ]:
WHO, TOOL = "T21", "torchvision"
p = df[df.person == WHO].iloc[0]
!anthro predict-photos --front "{p.front}" --side "{p.side}" --height {p.height_cm} --weight {p.weight_kg} --sex {p.sex} --age-range {p.age_range} --ckpt {RUN}/best.pt --tool {TOOL} --data {DATA_DIR}
print("tape:", p[[c for c in df.columns if c.endswith(("girth", "width", "length"))]].dropna().to_dict())

## Before saving
**Edit → Clear all outputs**, then save. CI refuses notebooks with saved outputs (POLICY 6).